# Week5_ex4 - Eddy Current Braking (Moving Magnet Over a Conductive Plate) [Full License]

Full (non-Student) license variant of `Week5_ex4.ipynb`. Same setup -- a permanent magnet moves at a constant velocity above a flat copper plate, and the goal is to see the eddy-current braking force induced in the plate as the magnet passes over it -- but this version assumes a full Ansys Electronics Desktop license instead of Student, so the Transient solver is actually available and this notebook can be run and verified end-to-end. Kept as a separate file so the Student-blocked version stays as-is for the portfolio narrative, and this one is ready to go whenever a full license is available. Path handling and project setup follow the same portable pattern used throughout this repo.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time
import matplotlib.pyplot as plt

In [ ]:
# 1. Launch AEDT -- assumes a full (non-Student) Ansys license, so student_version is hardcoded False
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=False)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type -- motion + induced eddy currents only exist under Transient
sol_type = "Transient"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week5_ex4_full_license"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week5_ex4_full_license"

In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
# Magnet material with N pole facing +z
magnet_mat = M3D.materials.duplicate_material("NdFe35", name="magnet")
magnet_mat.get_magnetic_coercivity()  # check current magnetization
magnet_mat.set_magnetic_coercivity(value=-890000, x=0, y=0, z=1)  # magnetized along +z

In [ ]:
# Copper plate the magnet will pass over
origin = [0, 0, 0]
sizes = [200, 150, 10]
Metal = M3D.modeler.create_box(origin, sizes, name="Metal", material="Copper")

# Bar magnet sitting just above the plate
origin = [80, 0, 10]
sizes = [20, 40, 5]
magnet = M3D.modeler.create_box(origin, sizes, name="Magnet", material=magnet_mat)
M3D.modeler.move(magnet, [10, 0, 0.1])

# Band wrapping the moving magnet -- pyaedt needs this to know what to translate
origin = [-1, -1, 10.05]
sizes = [42, 152, 10.1]
band = M3D.modeler.create_box(origin, sizes, name="Band", material="Vacuum")
M3D.modeler.move(band, [80, 0, 0])

region = M3D.modeler.create_region(pad_value=[500, 500, 500, 500, 500, 500], pad_type='Absolute Offset', name='Region')

In [ ]:
# Transient setup
setup = M3D.create_setup(name="MySetupAuto", setup_type="Transient")

setup.props["StopTime"] = "3s"
setup.props["TimeStep"] = "(1/100)s"
setup.props["SaveFieldsType"] = "Every N Steps"
setup.props["N Steps"] = "1s"
setup.props["Steps From"] = "0s"
setup.props["Steps To"] = "3s"

In [ ]:
meshing = M3D.mesh.assign_length_mesh(["Magnet", "Metal"], inside_selection=True, maximum_length=8, maximum_elements=None, name="Mesh")

In [ ]:
M3D.mesh.generate_mesh(setup.name)

In [ ]:
Motionsetup = M3D.assign_translate_motion("Band", coordinate_system='Global', axis='Y', positive_movement=True, start_position=0, periodic_translate=False, negative_limit=0,
                                           positive_limit=460, velocity="7cm_per_sec", mechanical_transient=False, mass=0, damping=0, load_force=0, motion_name="Motionsetup")

In [ ]:
# NOTE: the Student-license version of this notebook hit "No conductors defined in
# active design" here even though Metal is Copper. Since this version can actually be
# run end-to-end on a full license, this is the first thing to check if it recurs --
# confirm Metal is registered as a solid conductor before this call runs.
effect = M3D.eddy_effects_on(Metal, enable_eddy_effects=True, enable_displacement_current=True)

In [ ]:
setup.analyze()

In [ ]:
## Export force data ##

oModule = oDesign.GetModule("ReportSetup")

# saved flat, next to the notebook -- same convention as Images/
csv_path = "Week5_ex4_force_data.csv"

oModule.ExportToFile("Force Plot 1", csv_path, False)

In [ ]:
## Load and plot the exported data ##

df = pd.read_csv(csv_path)
print(df.head())  # sanity check that it loaded correctly

# Auto-detect columns (first = time, second = force)
time_column = df.columns[0]
force_column = df.columns[1]

time_vals = df[time_column]
force_vals = df[force_column]

os.makedirs("Images", exist_ok=True)

plt.figure(figsize=(10, 5))
plt.plot(time_vals, force_vals, color='r', linestyle='-', marker='o', markersize=3, label=force_column)
plt.xlabel(time_column)
plt.ylabel(force_column)
plt.title("Force vs Time Plot")
plt.legend()
plt.grid(True)
plt.savefig("Images/Week5_ex4_force_plot.png", dpi=150)
plt.show()

In [ ]:
## Save project ##

M3D.save_project()